# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdulmoiz-25/FlyRank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of Analysis + Time Window

**Lane:** Refresh / Content Opportunity Scoring

### Unit of Analysis

One row represents the daily performance of a single content item for one pseudonymized client on a specific report date.

The combination of **report_date**, **client_hash_id**, and **content_hash_id** uniquely identifies each observation.

### Development Time Window

To avoid using future information during development, this notebook uses the **March 2026** partition of the warehouse. The final month is intentionally excluded so it can remain an unseen evaluation period.

The following queries verify that this definition is correct.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---------------------------------------------------------
# Section 1
# Verify the unit of analysis and development time window
# ---------------------------------------------------------

import duckdb
from google.colab import userdata
import pandas as pd

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN secret was not found.")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
TYPE HUGGINGFACE,
TOKEN '{HF_TOKEN}'
);
""")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"

MARCH_DATA = (
    f"{WAREHOUSE}/fact_content_daily_performance/"
    "month=2026-03/*.parquet"
)

print("Warehouse connected successfully.")
print("Development month: March 2026")

Warehouse connected successfully.
Development month: March 2026


### Verification Query 1

In [20]:
contract_check = con.sql(f"""

SELECT

COUNT(*) AS total_rows,

COUNT(
DISTINCT (
report_date,
client_hash_id,
content_hash_id
)
) AS unique_rows,

MIN(report_date) AS first_day,

MAX(report_date) AS last_day

FROM read_parquet('{MARCH_DATA}')

""").df()

display(contract_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_rows,first_day,last_day
0,9841378,9841378,2026-03-01,2026-03-31


### Interpretation

The verification query confirms the proposed data contract.

If **total_rows** equals **unique_rows**, then every observation represents one unique combination of report date, client, and content item.

The reported minimum and maximum dates confirm that only the March 2026 development partition is being analyzed.

# 2. Fields: Feature / Label / Context / Excluded

This section classifies the fields used in the Refresh / Content Opportunity Scoring workflow. Each field is assigned a role based on whether it can be safely used before the prediction is made.

## Feature Fields

These variables are available at the decision point and may help estimate whether content requires refreshing.

| Field | Reason |
|-------|--------|
| gsc_impressions | Indicates how often a page appears in Google Search results. |
| gsc_clicks | Measures the amount of search traffic received. |
| gsc_avg_position | Represents the average ranking position in search results. |
| ga4_sessions | Measures website sessions associated with the content. |
| scroll_events | Indicates user engagement with the page. |

---

## Label

For this notebook, the prediction target is whether a content item should be considered a refresh opportunity based on historical performance. The label represents the outcome to be predicted and is **never used as an input feature**.

---

## Context Fields

These fields identify or filter records but are not predictive variables.

- report_date
- client_hash_id
- content_hash_id
- client_has_gsc
- client_has_ga4
- gsc_data_available
- ga4_data_available

---

## Excluded Fields

The following information is intentionally excluded.

| Field Type | Reason |
|------------|--------|
| Future performance | Not available at prediction time. |
| Target-derived variables | Would introduce label leakage. |
| Post-decision metrics | Created after the prediction date. |
| Client identity information | Used only for identification, not prediction. |

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Display the schema for the March 2026 partition
# ---------------------------------------------------------
# Section 2
# Inspect available fields
# ---------------------------------------------------------

schema = con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet('{MARCH_DATA}')
""").df()

display(schema)

print(f"Total available columns: {len(schema)}")

feature_fields = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "scroll_events"
]

context_fields = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "client_has_gsc",
    "client_has_ga4",
    "gsc_data_available",
    "ga4_data_available"
]

print("Feature Fields")
for f in feature_fields:
    print("•", f)

print()

print("Context Fields")
for c in context_fields:
    print("•", c)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


Total available columns: 31
Feature Fields
• gsc_impressions
• gsc_clicks
• gsc_avg_position
• ga4_sessions
• scroll_events

Context Fields
• report_date
• client_hash_id
• content_hash_id
• client_has_gsc
• client_has_ga4
• gsc_data_available
• ga4_data_available


# 3. Verify it with Queries

The following SQL queries verify the claims made in the data contract instead of relying on assumptions. They confirm the unit of analysis, validate the selected development window, and measure the availability of Search Console data using `IS TRUE`.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ---------------------------------------------------------
# Query 1: Verify the grain
# ---------------------------------------------------------

grain_df = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT (report_date, client_hash_id, content_hash_id)) AS unique_rows
FROM read_parquet('{MARCH_DATA}');
""").df()

display(grain_df)

if grain_df.loc[0, "total_rows"] == grain_df.loc[0, "unique_rows"]:
    print("✅ Grain verified: each row represents one report_date + client + content combination.")
else:
    print("⚠️ Duplicate combinations found.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_rows
0,9841378,9841378


✅ Grain verified: each row represents one report_date + client + content combination.


The total number of rows matches the number of unique `(report_date, client_hash_id, content_hash_id)` combinations, confirming the stated unit of analysis.

In [24]:
# ---------------------------------------------------------
# Query 2: Verify the March 2026 partition
# ---------------------------------------------------------

window_df = con.sql(f"""
SELECT
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date,
    COUNT(*) AS total_rows
FROM read_parquet('{MARCH_DATA}');
""").df()

display(window_df)

,first_date,last_date,total_rows
0,2026-03-01,2026-03-31,9841378


The minimum and maximum dates confirm that the notebook is using the intended March 2026 development partition. The row count provides a reference for the size of the working dataset.

In [25]:
# ---------------------------------------------------------
# Query 3: Search Console availability
# ---------------------------------------------------------

availability_df = con.sql(f"""
SELECT
    COUNT(*) AS rows_with_gsc,
    ROUND(
        100.0 * COUNT(*) /
        (SELECT COUNT(*) FROM read_parquet('{MARCH_DATA}')),
        2
    ) AS percent_of_rows
FROM read_parquet('{MARCH_DATA}')
WHERE gsc_data_available IS TRUE;
""").df()

display(availability_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows_with_gsc,percent_of_rows
0,3611061,36.69


This query counts the rows where Google Search Console data is available using `IS TRUE`, satisfying the assignment requirement and showing how much of the development data can support search-based modeling.

# 4. Data Limits

Although the warehouse provides rich historical search and analytics data, it has several important limitations that should be considered before modeling.

### 1. Unbalanced Client History

Different clients entered the warehouse at different times, so the amount of historical data varies across clients.

---

### 2. Partial Data Availability

Some observations contain only Google Search Console data or only Google Analytics data because platform access began on different dates.

---

### 3. Pseudonymized Warehouse

Client identifiers and content identifiers are hashed. This protects privacy but prevents business-level interpretation.

---

### 4. Historical Observations Only

The warehouse records what happened in the past. It cannot explain why rankings or traffic changed.

---

### 5. Development Window

This notebook intentionally develops on the March 2026 partition so the final month remains untouched for future evaluation.

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ---------------------------------------------------------
# Dataset summary
# ---------------------------------------------------------

summary = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT client_hash_id) AS total_clients,
    COUNT(DISTINCT content_hash_id) AS total_content,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet('{MARCH_DATA}');
""").df()

display(summary)

,total_rows,total_clients,total_content,first_date,last_date
0,9841378,55,331437,2026-03-01,2026-03-31


## Self-check

Before you submit, confirm each line honestly:

- [✔] Every section above is filled — markdown thinking AND the code that backs it
- [✔] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔] No client names, URLs, or private queries anywhere
- [✔] My claims use careful words: observed, measured, directional, decision-support
- [✔] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.